In [1]:
# Данный ноутбук использовал окружение google-colab
%pip install catboost fasttext -q

Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Building wheel for fasttext (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [133 lines of output]
      C:\Users\aslan\AppData\Local\Temp\pip-build-env-y4q22lpb\overlay\Lib\site-packages\setuptools\dist.py:599: SetuptoolsDeprecationWarning: Invalid dash-separated key 'description-file' in 'metadata' (setup.cfg), please use the underscore name 'description_file' instead.
      !!
      
              ********************************************************************************
              Usage of dash-separated 'description-file' will not be supported in future
              versions. Please use the underscore name 'description_file' instead.
              (Affected: fasttext).
      
              By 2026-Mar-03, you need to update your project and remove deprecated calls
              or your builds will no longer be supported.
      
              See https://setuptools.pypa.io/en/latest/userguide/declarative_config

# Домашнее задание "NLP. Часть 1"

In [27]:
import math
import re
import os
import random
import json
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Any

import torch
import numpy as np
import datasets
from transformers import BertTokenizer, BertModel

In [2]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(42)

In [3]:
def normalize_pretokenize_text(text: str) -> List[str]:
    text = text.lower()
    words = re.findall(r'\b\w+\b', text)
    return words

In [4]:
# This block is for tests only
test_corpus = [
    "the quick brown fox jumps over the lazy dog",
    "never jump over the lazy dog quickly",
    "brown foxes are quick and dogs are lazy"
]

def build_vocab(texts: List[str]) -> Tuple[List[str], Dict[str, int]]:
    all_words = []
    for text in texts:
        words = normalize_pretokenize_text(text)
        all_words.extend(words)
    vocab = sorted(set(all_words))
    vocab_index = {word: idx for idx, word in enumerate(vocab)}
    return vocab, vocab_index

vocab, vocab_index = build_vocab(test_corpus)

## Задание 1 (0.5 балла)
Реализовать One-Hot векторизацию текстов

In [5]:
def one_hot_vectorization(
    text: str,
    vocab: List[str] = None,
    vocab_index: Dict[str, int] = None
) -> List[List[int]]:
    if vocab is None or vocab_index is None:
        raise ValueError("vocab and vocab_index must be provided")

    words = normalize_pretokenize_text(text)
    one_hot_vectors = []

    for word in words:
        vector = [0] * len(vocab)
        if word in vocab_index:
            idx = vocab_index[word]
            vector[idx] = 1
        one_hot_vectors.append(vector)

    return one_hot_vectors

def test_one_hot_vectorization(
    vocab: List[str],
    vocab_index: Dict[str, int]
) -> bool:
    try:
        text = "the quick brown fox"
        result = one_hot_vectorization(text, vocab, vocab_index)

        if not isinstance(result, list):
            return False

        expected_length = len(vocab)
        if len(result[0]) != expected_length:
            return False

        words_in_text = normalize_pretokenize_text(text)
        for i, word in enumerate(words_in_text):
            if word in vocab_index:
                idx = vocab_index[word]
                if result[i][idx] != 1:
                    return False

        print("One-Hot-Vectors test PASSED")

        return True
    except Exception as e:
        print(f"One-Hot-Vectors test FAILED: {e}")
        return False

In [6]:
assert test_one_hot_vectorization(vocab, vocab_index)

One-Hot-Vectors test PASSED


## Задание 2 (0.5 балла)
Реализовать Bag-of-Words

In [7]:
def bag_of_words_vectorization(text: str) -> Dict[str, int]:
    words = normalize_pretokenize_text(text)
    bow_vector = Counter(words)
    return dict(bow_vector)

def test_bag_of_words_vectorization() -> bool:
    try:
        text = "the the quick brown brown brown"
        result = bag_of_words_vectorization(text)

        if not isinstance(result, dict):
            return False

        if result.get('the', 0) != 2:
            return False
        if result.get('quick', 0) != 1:
            return False
        if result.get('brown', 0) != 3:
            return False
        if result.get('nonexistent', 0) != 0:
            return False

        print("Bad-of-Words test PASSED")
        return True
    except Exception as e:
        print(f"Bag-of-Words test FAILED: {e}")
        return False

In [8]:
assert test_bag_of_words_vectorization()

Bad-of-Words test PASSED


## Задание 3 (0.5 балла)
Реализовать TF-IDF

In [10]:
def tf_idf_vectorization(
        text: str, 
        corpus: List[str] = None, 
        vocab: List[str] = None, 
        vocab_index: Dict[str, int] = None
        ) -> List[float]:
    if corpus is None or vocab is None or vocab_index is None:
        raise ValueError("Нужно передать corpus, vocab и vocab_index")

    n_docs = len(corpus)
    df = [0] * len(vocab)
    for doc in corpus:
        tokens = set(normalize_pretokenize_text(doc))
        for tok in tokens:
            idx = vocab_index.get(tok)
            if idx is not None:
                df[idx] += 1

    idf = [math.log((1 + n_docs) / (1 + df_i)) + 1.0 for df_i in df]

    words = normalize_pretokenize_text(text)
    counts = Counter(words)

    tf = [0.0] * len(vocab)
    for w, c in counts.items():
        idx = vocab_index.get(w)
        if idx is not None:
            tf[idx] = float(c)

    tf_idf = [tf[i] * idf[i] for i in range(len(vocab))]
    return tf_idf

def test_tf_idf_vectorization(corpus, vocab, vocab_index) -> bool:
    try:
        text = "the quick brown"
        result = tf_idf_vectorization(text, corpus, vocab, vocab_index)

        if not isinstance(result, list):
            return False

        expected_length = len(vocab)
        if len(result) != expected_length:
            return False

        for val in result:
            if not isinstance(val, float):
                return False

        print("TF-IDF test PASSED")
        return True
    except Exception as e:
        print(f"TF-IDF test FAILED: {e}")
        return False

In [11]:
assert test_tf_idf_vectorization(test_corpus, vocab, vocab_index)

TF-IDF test PASSED


## Задание 4 (1 балл)
Реализовать Positive Pointwise Mutual Information (PPMI).  
https://en.wikipedia.org/wiki/Pointwise_mutual_information
$$PPMI(word, context) = max(0, PMI(word, context))$$
$$PMI(word, context) = log \frac{P(word, context)}{P(word) P(context)} = log \frac{N(word, context)|(word, context)|}{N(word) N(context)}$$
где $N(word, context)$ -- число вхождений слова $word$ в окно $context$ (размер окна -- гиперпараметр)

In [ ]:
def ppmi_vectorization(
    text: str,
    corpus: List[str] = None,
    vocab: List[str] = None,
    vocab_index: Dict[str, int] = None,
    window_size: int = 2
) -> List[float]:
    if corpus is None or vocab is None or vocab_index is None:
        raise ValueError("Нужно передать corpus, vocab и vocab_index")

    word_counts = Counter()
    pair_counts = defaultdict(int)
    total_words = 0

    for doc in corpus:
        words = normalize_pretokenize_text(doc)
        total_words += len(words)
        word_counts.update(words)

        for i, w in enumerate(words):
            for j in range(max(0, i - window_size), min(len(words), i + window_size + 1)):
                if i == j:
                    continue
                pair = (w, words[j])
                pair_counts[pair] += 1

    total_pairs = sum(pair_counts.values())

    p_word = {w: word_counts[w] / total_words for w in vocab}

    def get_ppmi(w, c):
        p_wc = pair_counts.get((w, c), 0) / total_pairs if total_pairs > 0 else 0
        if p_wc == 0 or p_word.get(w, 0) == 0 or p_word.get(c, 0) == 0:
            return 0.0
        pmi = math.log2(p_wc / (p_word[w] * p_word[c]))
        return max(pmi, 0)

    words = normalize_pretokenize_text(text)
    vec = [0.0] * len(vocab)

    for i, v in enumerate(vocab):
        s = 0
        for w in words:
            s += get_ppmi(w, v)
        if len(words) > 0:
            vec[i] = s / len(words)
        else:
            vec[i] = 0.0

    return vec

def test_ppmi_vectorization(corpus, vocab, vocab_index) -> bool:
    try:
        text = "quick brown fox"
        result = ppmi_vectorization(text, corpus, vocab, vocab_index)

        if not isinstance(result, list):
            return False

        expected_length = len(vocab)
        if len(result) != expected_length:
            return False

        for val in result:
            if not isinstance(val, float):
                return False

        print("PPMI test PASSED")
        return True
    except Exception as e:
        print(f"PPMI test FAILED: {e}")
        return False

In [15]:
assert test_ppmi_vectorization(test_corpus, vocab, vocab_index)

PPMI test PASSED


In [16]:
vocab_index

{'and': 0,
 'are': 1,
 'brown': 2,
 'dog': 3,
 'dogs': 4,
 'fox': 5,
 'foxes': 6,
 'jump': 7,
 'jumps': 8,
 'lazy': 9,
 'never': 10,
 'over': 11,
 'quick': 12,
 'quickly': 13,
 'the': 14}

## Задание 5 (1 балл)
Реализовать получение эмбеддингов из fasttext и bert (для bert лучше использовать CLS токен)

In [21]:
from gensim.models import FastText

In [ ]:
def get_fasttext_embeddings(text: str, model_path: str = None, model: any = None) -> List[np.ndarray]:
    if model is None:
        if model_path is None:
            raise ValueError("Нужно передать путь к модели или саму модель")
        model = FastText.load(model_path)

    words = normalize_pretokenize_text(text)

    vectors = []
    for w in words:
        if w in model.wv:
            vec = model.wv[w]
        else:
            vec = np.zeros(model.vector_size)
        vectors.append(vec)

    return vectors

In [23]:
corpus = [
    "the quick brown fox jumps over the lazy dog".split(),
    "never jump over the lazy dog quickly".split()
]

ft_model = FastText(sentences=corpus, vector_size=50, window=3, min_count=1)

text = "quick brown fox"
vecs = get_fasttext_embeddings(text, model=ft_model)

print(f"Количество слов: {len(vecs)}")
print(f"Размер одного эмбеддинга: {vecs[0].shape}")
print(vecs[0][:5])

Количество слов: 3
Размер одного эмбеддинга: (50,)
[-0.00312437  0.00094703 -0.00022899 -0.00206479 -0.00363089]


In [28]:
def get_bert_embeddings(
    text: str,
    model_name: str = 'bert-base-uncased',
    pool_method: str = 'cls'
) -> np.ndarray:
    tokenizer = BertTokenizer.from_pretrained(model_name)
    model = BertModel.from_pretrained(model_name)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)

    with torch.no_grad():
        outputs = model(**inputs)
        last_hidden_state = outputs.last_hidden_state
        if pool_method == 'cls':
            embeddings = last_hidden_state[:, 0, :]


    return embeddings.squeeze().numpy()

In [31]:
text = "The quick brown fox jumps over the lazy dog"

emb_cls = get_bert_embeddings(text)

print(emb_cls.shape)

(768,)


## Задание 6 (1.5 балла)
Реализовать обучение так, чтобы можно было поверх эмбеддингов, реализованных в предыдущих заданиях, обучить какую-то модель (вероятно неглубокую, например, CatBoost) на задаче классификации текстов ([IMDB](https://huggingface.co/datasets/stanfordnlp/imdb)).

In [ ]:
from typing import List, Dict, Tuple
import os, random
import numpy as np
from gensim.models import FastText

class SimpleVectorizer:
    def __init__(self, method="bow", ft_size=100, ft_window=5, ft_min_count=2):
        self.method = method
        self.vocab = None
        self.vocab_index = None
        self.train_corpus = None
        self.ft_model = None
        self.ft_size = ft_size
        self.ft_window = ft_window
        self.ft_min_count = ft_min_count

    def _build_vocab(self, texts: List[str]):
        all_words = []
        for t in texts:
            all_words.extend(normalize_pretokenize_text(t))
        vocab = sorted(set(all_words))
        vocab_index = {w:i for i, w in enumerate(vocab)}
        return vocab, vocab_index

    def fit(self, texts: List[str]):
        self.train_corpus = list(texts)

        if self.method in {"one_hot", "bow", "tfidf", "ppmi"}:
            self.vocab, self.vocab_index = self._build_vocab(self.train_corpus)

        if self.method == "fasttext":
            toks = [normalize_pretokenize_text(t) for t in self.train_corpus]
            self.ft_model = FastText(
                sentences=toks, vector_size=self.ft_size,
                window=self.ft_window, min_count=self.ft_min_count, workers=4, sg=1
            )

    def transform(self, texts: List[str]) -> List[List[float]]:
        X = []
        for text in texts:
            if self.method == "bow":
                bow = bag_of_words_vectorization(text)
                X.append([bow.get(w, 0) for w in self.vocab])

            elif self.method == "one_hot":
                mats = one_hot_vectorization(text, self.vocab, self.vocab_index)
                doc_vec = [0]*len(self.vocab)
                for row in mats:
                    for j, v in enumerate(row):
                        if v:
                            doc_vec[j] += 1
                X.append(doc_vec)

            elif self.method == "tfidf":
                vec = tf_idf_vectorization(text, self.train_corpus, self.vocab, self.vocab_index)
                X.append(vec)

            elif self.method == "ppmi":
                vec = ppmi_vectorization(text, self.train_corpus, self.vocab, self.vocab_index)
                X.append(vec)

            elif self.method == "fasttext":
                embs = get_fasttext_embeddings(text, model=self.ft_model)
                if embs:
                    X.append(np.mean(embs, axis=0).tolist())
                else:
                    X.append([0.0]*self.ft_model.vector_size)

            elif self.method == "bert":
                X.append(get_bert_embeddings(text).tolist())

            else:
                raise ValueError(f"unknown method: {self.method}")
        return X


In [61]:
import datasets, random
random.seed(42)

def load_imdb_split(split="train", sample_size=50, balance=True):
    ds = datasets.load_dataset("imdb", split=split)
    texts_all, labels_all = [], []
    for it in ds:
        t = it.get("text", "")
        if isinstance(t, str) and t.strip():
            texts_all.append(t)
            labels_all.append(int(it["label"]))

    if split == "train" and balance:
        idx_pos = [i for i,y in enumerate(labels_all) if y==1]
        idx_neg = [i for i,y in enumerate(labels_all) if y==0]
        take = sample_size//2
        take = max(1, min(take, len(idx_pos), len(idx_neg)))
        sel = random.sample(idx_pos, take) + random.sample(idx_neg, take)
        random.shuffle(sel)
        texts = [texts_all[i] for i in sel]
        labels = [labels_all[i] for i in sel]
    else:
        if sample_size:
            texts = texts_all[:sample_size]
            labels = labels_all[:sample_size]
        else:
            texts, labels = texts_all, labels_all

    return texts, labels


In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

def train_simple(method="bow", train_size=2000, test_size=2000, val_size=0.2):
    tr_texts, tr_labels = load_imdb_split("train", sample_size=train_size, balance=True)
    te_texts, te_labels = load_imdb_split("test", sample_size=test_size, balance=False)

    vec = SimpleVectorizer(method=method)
    vec.fit(tr_texts)

    X_tr_full = vec.transform(tr_texts)
    X_te = vec.transform(te_texts)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_tr_full, tr_labels, test_size=val_size, random_state=42, stratify=tr_labels
    )

    clf = CatBoostClassifier(
        iterations=300, depth=6, learning_rate=0.1,
        loss_function="Logloss", eval_metric="AUC",
        verbose=False, allow_writing_files=False
    )
    clf.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False)

    y_pred = clf.predict(X_te)
    acc = accuracy_score(te_labels, y_pred)
    f1m = f1_score(te_labels, y_pred, average="macro")

    print(f"\n=== {method.upper()} ===")
    print(f"Test Acc: {acc:.4f} | Macro-F1: {f1m:.4f}")
    print(classification_report(te_labels, y_pred, digits=4))

    return clf, vec


In [63]:
for m in ["bow", "one_hot", "tfidf", "ppmi", "fasttext", "bert"]:
    clf, vec = train_simple(method=m, train_size=50, test_size=50)



=== BOW ===
Test Acc: 0.5400 | Macro-F1: 0.3506
              precision    recall  f1-score   support

           0     1.0000    0.5400    0.7013        50
           1     0.0000    0.0000    0.0000         0

    accuracy                         0.5400        50
   macro avg     0.5000    0.2700    0.3506        50
weighted avg     1.0000    0.5400    0.7013        50



C:\Users\aslan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\aslan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\aslan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_classification.py:1565: Unde


=== ONE_HOT ===
Test Acc: 0.5600 | Macro-F1: 0.3590
              precision    recall  f1-score   support

           0     1.0000    0.5600    0.7179        50
           1     0.0000    0.0000    0.0000         0

    accuracy                         0.5600        50
   macro avg     0.5000    0.2800    0.3590        50
weighted avg     1.0000    0.5600    0.7179        50



C:\Users\aslan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\aslan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\aslan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_classification.py:1565: Unde


=== TFIDF ===
Test Acc: 0.7000 | Macro-F1: 0.4118
              precision    recall  f1-score   support

           0     1.0000    0.7000    0.8235        50
           1     0.0000    0.0000    0.0000         0

    accuracy                         0.7000        50
   macro avg     0.5000    0.3500    0.4118        50
weighted avg     1.0000    0.7000    0.8235        50



...

C:\Users\aslan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\aslan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\aslan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_classification.py:1565: Unde


=== FASTTEXT ===
Test Acc: 0.6800 | Macro-F1: 0.4048
              precision    recall  f1-score   support

           0     1.0000    0.6800    0.8095        50
           1     0.0000    0.0000    0.0000         0

    accuracy                         0.6800        50
   macro avg     0.5000    0.3400    0.4048        50
weighted avg     1.0000    0.6800    0.8095        50



C:\Users\aslan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\aslan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\aslan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_classification.py:1565: Unde


=== BERT ===
Test Acc: 0.6400 | Macro-F1: 0.3902
              precision    recall  f1-score   support

           0     1.0000    0.6400    0.7805        50
           1     0.0000    0.0000    0.0000         0

    accuracy                         0.6400        50
   macro avg     0.5000    0.3200    0.3902        50
weighted avg     1.0000    0.6400    0.7805        50



C:\Users\aslan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\aslan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\aslan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\metrics\_classification.py:1565: Unde